In [ ]:
# Análisis de Resistencia Genética en E. coli y Generación de Informes Automatizados
#Estudiante: Masiel Aguilar Ameller
#Codigo: 87770

import os
import glob
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import nt_search
import pandas as pd
from datetime import datetime
import re
from pathlib import Path
from docx import Document
from docx.shared import Inches, Pt
from docx.oxml.ns import qn
from docx.enum.section import WD_HEADER_FOOTER
from docx.oxml import OxmlElement
from PIL import Image
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
from docx.shared import RGBColor
from docx.shared import Inches
from docx.shared import Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT

# Configuración de directorios
GENOMAS_DIR = "genomas_ecoli"
GENES_DIR = "genes_resistentes"
OUTPUT_DIR = "informes_generados"
TEMPLATE_FILE = "informe_bacteria_XXXX.docx"

# Crear directorio de salida si no existe
os.makedirs(OUTPUT_DIR, exist_ok=True)

def cargar_genes_resistencia():
    """
    Carga las secuencias de los genes de resistencia desde archivos FASTA
    """
    genes_resistencia = {}
    
    # Cargar blatem1
    blatem1_path = os.path.join(GENES_DIR, "blatem1.fasta")
    if os.path.exists(blatem1_path):
        for record in SeqIO.parse(blatem1_path, "fasta"):
            genes_resistencia["blatem1"] = str(record.seq)
            break
    
    # Cargar blactxm15
    blactxm15_path = os.path.join(GENES_DIR, "blactxm15.fasta")
    if os.path.exists(blactxm15_path):
        for record in SeqIO.parse(blactxm15_path, "fasta"):
            genes_resistencia["blactxm15"] = str(record.seq)
            break
    
    return genes_resistencia

def buscar_gen_en_secuencia(secuencia_bacteria, secuencia_gen, umbral_similitud=0.8):
    """
    Busca un gen de resistencia en la secuencia de la bacteria
    Utiliza búsqueda optimizada por subsecuencias
    """
    secuencia_bacteria = str(secuencia_bacteria).upper()
    secuencia_gen = str(secuencia_gen).upper()
    
    # Búsqueda exacta primero (más rápida)
    if secuencia_gen in secuencia_bacteria:
        return True, "EXACTA"
    
    # Búsqueda de subsecuencias más pequeñas para acelerar
    len_gen = len(secuencia_gen)
    if len_gen == 0:
        return False, "GEN_VACIO"
    
    # Si el gen es muy largo, buscar por fragmentos
    if len_gen > 500:
        # Dividir en fragmentos de 100 nucleótidos
        fragmentos = [secuencia_gen[i:i+100] for i in range(0, len_gen, 100)]
        fragmentos_encontrados = 0
        
        for fragmento in fragmentos:
            if fragmento in secuencia_bacteria:
                fragmentos_encontrados += 1
        
        proporcion_fragmentos = fragmentos_encontrados / len(fragmentos)
        if proporcion_fragmentos >= 0.7:  # Al menos 70% de fragmentos encontrados
            return True, f"FRAGMENTOS_{proporcion_fragmentos:.2%}"
    
    # Búsqueda con ventana deslizante optimizada (solo cada 10 posiciones)
    mejor_coincidencia = 0
    step = max(1, len_gen // 100)  # Saltar posiciones para acelerar
    
    for i in range(0, len(secuencia_bacteria) - len_gen + 1, step):
        ventana = secuencia_bacteria[i:i + len_gen]
        coincidencias = sum(1 for a, b in zip(ventana, secuencia_gen) if a == b)
        similitud = coincidencias / len_gen
        
        if similitud > mejor_coincidencia:
            mejor_coincidencia = similitud
            
        # Si ya encontramos una buena coincidencia, no seguir buscando
        if similitud >= umbral_similitud:
            return True, f"PARCIAL_{similitud:.2%}"
    
    if mejor_coincidencia >= umbral_similitud:
        return True, f"PARCIAL_{mejor_coincidencia:.2%}"
    
    return False, f"NO_ENCONTRADO_{mejor_coincidencia:.2%}"

def analizar_bacteria(archivo_bacteria, genes_resistencia):
    """
    Analiza una bacteria individual para detectar genes de resistencia
    """
    resultados = {
        'nombre_bacteria': os.path.splitext(os.path.basename(archivo_bacteria))[0],
        'archivo': archivo_bacteria,
        'blatem1': {'presente': False, 'tipo': 'NO_ENCONTRADO'},
        'blactxm15': {'presente': False, 'tipo': 'NO_ENCONTRADO'},
        'secuencia_length': 0
    }
    
    try:
        # Intentar diferentes formatos de archivo
        formatos = ["fasta", "genbank", "embl"]
        secuencia_bacteria = None
        
        for formato in formatos:
            try:
                for record in SeqIO.parse(archivo_bacteria, formato):
                    secuencia_bacteria = record.seq
                    resultados['secuencia_length'] = len(secuencia_bacteria)
                    break
                if secuencia_bacteria:
                    break
            except:
                continue
        
        if secuencia_bacteria is None:
            print(f"    ⚠️  No se pudo leer la secuencia de {os.path.basename(archivo_bacteria)}")
            return resultados
            
        # Buscar cada gen de resistencia
        for gen_nombre, gen_secuencia in genes_resistencia.items():
            presente, tipo = buscar_gen_en_secuencia(secuencia_bacteria, gen_secuencia)
            resultados[gen_nombre]['presente'] = presente
            resultados[gen_nombre]['tipo'] = tipo
            
    except Exception as e:
        print(f"    ❌ Error al procesar {archivo_bacteria}: {e}")
    
    return resultados

def generar_informe_word(resultado_bacteria, fecha_actual):
    """
    Genera un informe en Word basado en la plantilla para cada bacteria
    """
    # Crear nuevo documento
    doc = Document()
    
    # Configurar página
    section = doc.sections[0]
    section.page_height = Inches(11.69)  # A4
    section.page_width = Inches(8.27)
    
    # Título principal
    titulo = doc.add_paragraph()
    titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = titulo.add_run("MODELO DE INFORME DE CORRELACIÓN")
    run.bold = True
    run.font.size = Pt(14)
    
    # Subtítulo
    subtitulo = doc.add_paragraph()
    subtitulo.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = subtitulo.add_run("CLÍNICA ADMINISTRATIVA DEL SUS")
    run.bold = True
    run.font.size = Pt(12)
    
    # Encabezado del memo
    doc.add_paragraph()
    encabezado_lines = [
        "A: DIRECTOR MUNICIPAL DE SALUD GAMLP",
        "GOBIERNO AUTÓNOMO MUNICIPAL DE LA PAZ",
        "VÍA: LABORATORIO FARMACOLÓGICO",
        "DE: RESPONSABLE DE LA ELABORACIÓN DEL INFORME",
        "BIOINGENIERO [Masiel Aguilar Ameller]",
        f"REF.: INFORME DE REVISIÓN DEL CUMPLIMIENTO DE INFORME GENÉTICO DE LA BACTERIA E. COLI {resultado_bacteria['nombre_bacteria'].upper()} REALIZADA EN EL HOSPITAL MUNICIPAL DE SEGUNDO NIVEL LA PORTADA CORRESPONDIENTE AL MES DE JUNIO DE LA GESTIÓN 2025"
    ]
    
    for line in encabezado_lines:
        para = doc.add_paragraph()
        run = para.add_run(line)
        run.bold = True
        run.font.size = Pt(10)
    
    # Fecha
    fecha_para = doc.add_paragraph()
    fecha_run = fecha_para.add_run(f"Fecha: {fecha_actual.strftime('%d de junio de 2025')}")
    fecha_run.bold = True
    
    # Saludo
    doc.add_paragraph("De mi mayor consideración:")
    
    # Párrafo introductorio
    intro_text = """En cumplimiento a lo establecido en el Reglamento para la aplicación técnica y la gestión administrativa y financiera de la Ley N° 1152, aprobado mediante Resolución Ministerial N° 0251 de 30 de junio de 2021, remito el informe de revisión del cumplimiento de la Correlación Clínica Administrativa de los Servicios de Salud otorgados a la población beneficiaria, en el Hospital Municipal de Segundo Nivel La Portada la gestión 2025."""
    
    doc.add_paragraph(intro_text)
    
    # Secciones del informe (manteniendo estructura original pero simplificada)
    secciones = [
        ("1. ANTECEDENTES", [
            "La Ley N° 475, de 30 de diciembre de 2013, de Prestaciones de Servicios de Salud Integral del Estado Plurinacional de Bolivia, establece la atención integral y protección financiera en salud.",
            "Las modificaciones posteriores han optimizado el uso de recursos y ampliado la cobertura hacia un Sistema Único de Salud Universal y Gratuito."
        ]),
        ("2. JUSTIFICACIÓN", [
            "La correlación clínica administrativa busca la coherencia entre diagnósticos, procedimientos y servicios declarados, con el propósito de mejorar la calidad del dato y la atención de pacientes.",
            f"El presente análisis genético de la bacteria E. coli {resultado_bacteria['nombre_bacteria']} se realiza para identificar posibles resistencias antibióticas que puedan afectar el tratamiento de pacientes."
        ]),
        ("3. METODOLOGÍA", [
            "Se realizó el análisis genético mediante secuenciación y búsqueda de genes de resistencia específicos.",
            "Se aplicaron técnicas bioinformáticas para la identificación de los genes blatem1 y blactxm15.",
            "Se utilizaron herramientas de análisis de secuencias con umbrales de similitud apropiados."
        ]),
        ("4. ANÁLISIS TÉCNICO", [
            f"Se analizó la secuencia genética de la bacteria E. coli {resultado_bacteria['nombre_bacteria']}",
            f"Longitud de secuencia analizada: {resultado_bacteria['secuencia_length']} nucleótidos",
            "Se realizó la búsqueda de genes de resistencia específicos mediante algoritmos bioinformáticos."
        ])
    ]
    
    for titulo_seccion, contenido in secciones:
        doc.add_paragraph()
        titulo_para = doc.add_paragraph()
        titulo_run = titulo_para.add_run(titulo_seccion)
        titulo_run.bold = True
        titulo_run.font.size = Pt(11)
        
        for item in contenido:
            doc.add_paragraph(item)
    
    # Resultados y Conclusiones
    doc.add_paragraph()
    resultados_titulo = doc.add_paragraph()
    resultados_run = resultados_titulo.add_run("RESULTADOS")
    resultados_run.bold = True
    resultados_run.font.size = Pt(11)
    
    # Crear tabla de resultados
    table = doc.add_table(rows=3, cols=3)
    table.style = 'Table Grid'
    
    # Encabezados
    headers = ['Gen de Resistencia', 'Presente', 'Tipo de Detección']
    for i, header in enumerate(headers):
        cell = table.cell(0, i)
        cell.text = header
        run = cell.paragraphs[0].runs[0]
        run.bold = True
    
    # Datos blatem1
    table.cell(1, 0).text = 'blatem1'
    table.cell(1, 1).text = 'SÍ' if resultado_bacteria['blatem1']['presente'] else 'NO'
    table.cell(1, 2).text = resultado_bacteria['blatem1']['tipo']
    
    # Datos blactxm15
    table.cell(2, 0).text = 'blactxm15'
    table.cell(2, 1).text = 'SÍ' if resultado_bacteria['blactxm15']['presente'] else 'NO'
    table.cell(2, 2).text = resultado_bacteria['blactxm15']['tipo']
    
    # Conclusiones
    doc.add_paragraph()
    conclusiones_titulo = doc.add_paragraph()
    conclusiones_run = conclusiones_titulo.add_run("CONCLUSIONES")
    conclusiones_run.bold = True
    conclusiones_run.font.size = Pt(11)
    
    # Conclusión para blatem1
    blatem1_resultado = "POSITIVO" if resultado_bacteria['blatem1']['presente'] else "NEGATIVO"
    conclusion1 = f"1. Se realizó la verificación de la posible resistencia de la bacteria {resultado_bacteria['nombre_bacteria']}. De la que se sospecha posee el gen de resistencia \"blatem1\", del análisis genético realizado se puede concluir que esta es:\n\n{blatem1_resultado}"
    doc.add_paragraph(conclusion1)
    
    # Conclusión para blactxm15
    blactxm15_resultado = "POSITIVO" if resultado_bacteria['blactxm15']['presente'] else "NEGATIVO"
    conclusion2 = f"2. Se realizó la verificación de la posible resistencia de la bacteria {resultado_bacteria['nombre_bacteria']}. De la que se sospecha posee el gen de resistencia \"blactxm15\", del análisis genético realizado se puede concluir que esta es:\n\n{blactxm15_resultado}"
    doc.add_paragraph(conclusion2)
    
    # Firma
    doc.add_paragraph()
    doc.add_paragraph("Ingeniero [Masiel Aguilar Ameller]")
    doc.add_paragraph("C.I.: [10931401 LP]")
    doc.add_paragraph()
    doc.add_paragraph("Cc. Archivo.")
    
    return doc

def main():
    """
    Función principal que ejecuta todo el análisis
    """
    print("=== ANÁLISIS DE RESISTENCIA GENÉTICA EN E. COLI ===")
    print(f"Fecha de análisis: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print()
    
    # 1. Cargar genes de resistencia
    print("1. Cargando genes de resistencia...")
    genes_resistencia = cargar_genes_resistencia()
    
    if not genes_resistencia:
        print("ERROR: No se pudieron cargar los genes de resistencia.")
        print(f"Verifique que existan los archivos en el directorio: {GENES_DIR}")
        return
    
    print(f"   - Genes cargados: {list(genes_resistencia.keys())}")
    for gen, seq in genes_resistencia.items():
        print(f"   - {gen}: {len(seq)} nucleótidos")
    print()
    
    # 2. Buscar archivos de genomas
    print("2. Buscando archivos de genomas...")
    
    # Buscar diferentes extensiones comunes para archivos de secuencias
    extensiones = ["*.fasta", "*.fa", "*.fna", "*.fas"]
    archivos_bacterias = []
    
    for extension in extensiones:
        patron_genomas = os.path.join(GENOMAS_DIR, extension)
        archivos_encontrados = glob.glob(patron_genomas)
        archivos_bacterias.extend(archivos_encontrados)
        if archivos_encontrados:
            print(f"   - Encontrados {len(archivos_encontrados)} archivos con extensión {extension}")
    
    if not archivos_bacterias:
        print(f"ERROR: No se encontraron archivos de genomas en {GENOMAS_DIR}")
        print("Extensiones buscadas: .fasta, .fa, .fna, .fas")
        print("Archivos disponibles en el directorio:")
        try:
            archivos_dir = os.listdir(GENOMAS_DIR)
            for archivo in archivos_dir[:10]:  # Mostrar primeros 10
                print(f"  - {archivo}")
            if len(archivos_dir) > 10:
                print(f"  ... y {len(archivos_dir) - 10} archivos más")
        except:
            print("  No se pudo acceder al directorio")
        return
    
    print(f"   - Encontrados {len(archivos_bacterias)} archivos de genomas")
    for archivo in archivos_bacterias[:5]:  # Mostrar solo los primeros 5
        print(f"     • {os.path.basename(archivo)}")
    if len(archivos_bacterias) > 5:
        print(f"     ... y {len(archivos_bacterias) - 5} más")
    print()
    
    # 3. Analizar bacterias (limitar a 8 como se solicita)
    print("3. Analizando bacterias...")
    resultados_analisis = []
    bacterias_a_procesar = archivos_bacterias[:8]  # Solo procesar 8 bacterias
    
    for i, archivo_bacteria in enumerate(bacterias_a_procesar, 1):
        print(f"   Procesando {i}/8: {os.path.basename(archivo_bacteria)}")
        print(f"   Progreso: {'█' * (i-1)}{'▓'}{'░' * (8-i)} {i*12.5:.1f}%")
        
        resultado = analizar_bacteria(archivo_bacteria, genes_resistencia)
        resultados_analisis.append(resultado)
        
        # Mostrar resultado resumido
        blatem1_status = "✓" if resultado['blatem1']['presente'] else "✗"
        blactxm15_status = "✓" if resultado['blactxm15']['presente'] else "✗"
        print(f"     blatem1: {blatem1_status} ({resultado['blatem1']['tipo']}) | blactxm15: {blactxm15_status} ({resultado['blactxm15']['tipo']})")
        print(f"     Secuencia: {resultado['secuencia_length']:,} nucleótidos")
        print()
    
    # 4. Generar informes
    print("4. Generando informes en Word...")
    fecha_actual = datetime.now()
    
    for i, resultado in enumerate(resultados_analisis, 1):
        print(f"   Generando informe {i}/8: {resultado['nombre_bacteria']}")
        
        try:
            doc = generar_informe_word(resultado, fecha_actual)
            nombre_archivo = f"informe_bacteria_{resultado['nombre_bacteria']}.docx"
            ruta_archivo = os.path.join(OUTPUT_DIR, nombre_archivo)
            doc.save(ruta_archivo)
            print(f"     ✓ Guardado: {nombre_archivo}")
            
        except Exception as e:
            print(f"     ✗ Error al generar informe para {resultado['nombre_bacteria']}: {e}")
    
    print()
    
    # 5. Resumen final
    print("5. RESUMEN FINAL")
    print("="*50)
    
    # Crear DataFrame para resumen
    df_resultados = pd.DataFrame([
        {
            'Bacteria': r['nombre_bacteria'],
            'blatem1': 'SÍ' if r['blatem1']['presente'] else 'NO',
            'blatem1_tipo': r['blatem1']['tipo'],
            'blactxm15': 'SÍ' if r['blactxm15']['presente'] else 'NO',
            'blactxm15_tipo': r['blactxm15']['tipo'],
            'Secuencia_length': r['secuencia_length']
        }
        for r in resultados_analisis
    ])
    
    print(df_resultados.to_string(index=False))
    print()
    
    # Estadísticas
    total_bacterias = len(resultados_analisis)
    con_blatem1 = sum(1 for r in resultados_analisis if r['blatem1']['presente'])
    con_blactxm15 = sum(1 for r in resultados_analisis if r['blactxm15']['presente'])
    con_ambos = sum(1 for r in resultados_analisis if r['blatem1']['presente'] and r['blactxm15']['presente'])
    
    print("ESTADÍSTICAS:")
    print(f"- Total de bacterias analizadas: {total_bacterias}")
    print(f"- Bacterias con gen blatem1: {con_blatem1} ({con_blatem1/total_bacterias*100:.1f}%)")
    print(f"- Bacterias con gen blactxm15: {con_blactxm15} ({con_blactxm15/total_bacterias*100:.1f}%)")
    print(f"- Bacterias con ambos genes: {con_ambos} ({con_ambos/total_bacterias*100:.1f}%)")
    print()
    
    # Guardar resumen en CSV
    csv_path = os.path.join(OUTPUT_DIR, f"resumen_analisis_{fecha_actual.strftime('%Y%m%d_%H%M%S')}.csv")
    df_resultados.to_csv(csv_path, index=False)
    print(f"Resumen guardado en: {csv_path}")
    print(f"Informes generados en directorio: {OUTPUT_DIR}")
    print()
    print("¡Análisis completado exitosamente!")

if __name__ == "__main__":
    main()

=== ANÁLISIS DE RESISTENCIA GENÉTICA EN E. COLI ===
Fecha de análisis: 2025-06-08 21:26:33

1. Cargando genes de resistencia...
   - Genes cargados: ['blatem1', 'blactxm15']
   - blatem1: 979 nucleótidos
   - blactxm15: 2315 nucleótidos

2. Buscando archivos de genomas...
   - Encontrados 8 archivos con extensión *.fna
   - Encontrados 8 archivos de genomas
     • GCF_050389795.1_MS10495_ARL02201_ST410_genomic.fna
     • GCF_050516345.1_ASM5051634v1_genomic.fna
     • GCF_050516355.1_ASM5051635v1_genomic.fna
     • GCF_050516365.1_ASM5051636v1_genomic.fna
     • GCF_050516375.1_ASM5051637v1_genomic.fna
     ... y 3 más

3. Analizando bacterias...
   Procesando 1/8: GCF_050389795.1_MS10495_ARL02201_ST410_genomic.fna
   Progreso: ▓░░░░░░░ 12.5%
     blatem1: ✗ (NO_ENCONTRADO_32.18%) | blactxm15: ✗ (NO_ENCONTRADO_29.76%)
     Secuencia: 3,106,818 nucleótidos

   Procesando 2/8: GCF_050516345.1_ASM5051634v1_genomic.fna
   Progreso: █▓░░░░░░ 25.0%
     blatem1: ✓ (FRAGMENTOS_80.00%) | blact